In [42]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import sklearn
import nflreadpy as nfl

# NFL Win Prediction - Data Exploration

This notebook explores the historical NFL dataset that will eventually be used to train a binary classification model.

The initial goals are to:

1. Load the required Python libraries.
2. Load historical NFL game data.
3. Understand the structure of the dataset.
4. Identify useful columns.
5. Identify potential sources of data leakage.

In [43]:
print("pands version:", pd.__version__)
print("numpy version:", np.__version__)
print("scikit-learn version:", sklearn.__version__)

pands version: 3.0.6
numpy version: 2.5.3
scikit-learn version: 1.9.1


In [44]:
import sys 

print(sys.executable)
print(sys.version)


c:\Users\owens\anaconda3\envs\nfl-ml\python.exe
3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


## Load Historical NFL Schedule Data

We will begin by loading historical NFL schedules and game results.

The goal at this stage is not to build a model yet. It is to understand the structure of the dataset and determine what information is available for each game.

In [45]:
seasons = list(range(2015, 2026))

schedules = nfl.load_schedules(seasons)

In [46]:
games = schedules.to_pandas()

In [47]:
games.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field


In [48]:
games.shape

(3028, 46)

In [49]:
games.columns.tolist()

['game_id',
 'season',
 'game_type',
 'week',
 'gameday',
 'weekday',
 'gametime',
 'away_team',
 'away_score',
 'home_team',
 'home_score',
 'location',
 'result',
 'total',
 'overtime',
 'old_game_id',
 'gsis',
 'nfl_detail_id',
 'pfr',
 'pff',
 'espn',
 'ftn',
 'away_rest',
 'home_rest',
 'away_moneyline',
 'home_moneyline',
 'spread_line',
 'away_spread_odds',
 'home_spread_odds',
 'total_line',
 'under_odds',
 'over_odds',
 'div_game',
 'roof',
 'surface',
 'temp',
 'wind',
 'away_qb_id',
 'home_qb_id',
 'away_qb_name',
 'home_qb_name',
 'away_coach',
 'home_coach',
 'referee',
 'stadium_id',
 'stadium']

In [50]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 3028 entries, 0 to 3027
Data columns (total 46 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   game_id           3028 non-null   str    
 1   season            3028 non-null   int32  
 2   game_type         3028 non-null   str    
 3   week              3028 non-null   int32  
 4   gameday           3028 non-null   str    
 5   weekday           3028 non-null   str    
 6   gametime          3028 non-null   str    
 7   away_team         3028 non-null   str    
 8   away_score        3028 non-null   int32  
 9   home_team         3028 non-null   str    
 10  home_score        3028 non-null   int32  
 11  location          3028 non-null   str    
 12  result            3028 non-null   int32  
 13  total             3028 non-null   int32  
 14  overtime          3028 non-null   int32  
 15  old_game_id       3028 non-null   str    
 16  gsis              3028 non-null   int32  
 17  nfl_de

In [51]:
games["season"].value_counts().sort_index

<bound method Series.sort_index of season
2021    285
2023    285
2024    285
2025    285
2022    284
2020    269
2015    267
2016    267
2017    267
2018    267
2019    267
Name: count, dtype: int64>

In [52]:
games.sample(5)

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
1601,2020_20_TB_GB,2020,CON,20,2021-01-24,Sunday,15:05,TB,31,GB,...,10.0,00-0019596,00-0023459,Tom Brady,Aaron Rodgers,Bruce Arians,Matt LaFleur,Clete Blakeman,GNB00,Lambeau Field
2257,2023_06_NO_HOU,2023,REG,6,2023-10-15,Sunday,13:00,NO,13,HOU,...,NaN,00-0031280,00-0039163,Derek Carr,C.J. Stroud,Dennis Allen,DeMeco Ryans,Alan Eck,HOU00,NRG Stadium
506,2016_16_DET_DAL,2016,REG,16,2016-12-26,Monday,20:30,DET,21,DAL,...,NaN,00-0026498,00-0033077,Matthew Stafford,Dak Prescott,Jim Caldwell,Jason Garrett,John Hussey,DAL00,AT&T Stadium
2277,2023_07_MIA_PHI,2023,REG,7,2023-10-22,Sunday,20:20,MIA,17,PHI,...,17.0,00-0036212,00-0036389,Tua Tagovailoa,Jalen Hurts,Mike McDaniel,Nick Sirianni,Brad Allen,PHI00,Lincoln Financial Field
2815,2025_05_TEN_ARI,2025,REG,5,2025-10-05,Sunday,16:05,TEN,22,ARI,...,NaN,00-0040676,00-0035228,Cam Ward,Kyler Murray,Brian Callahan,Jonathan Gannon,Shawn Smith,PHO00,State Farm Stadium


## Build the Game Backbone

The schedule dataset will serve as the backbone of the project.

Each row represents one NFL game. Additional team, player, matchup, and contextual features will later be attached to these rows using identifiers such as season, week, team, and game ID.

Before building features, the schedule data must be cleaned and the prediction target must be defined.

In [53]:
games["game_type"].value_counts()

game_type
REG    2895
WC       56
DIV      44
CON      22
SB       11
Name: count, dtype: int64

In [54]:
regular_games = games.loc[games["game_type"] == "REG"].copy()

In [55]:
games.loc[...]

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
1,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
2,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
3,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3023,2025_20_LA_CHI,2025,DIV,20,2026-01-18,Sunday,18:30,LA,20,CHI,...,6.0,00-0026498,00-0039918,Matthew Stafford,Caleb Williams,Sean McVay,Ben Johnson,Shawn Hochuli,CHI98,Soldier Field
3024,2025_20_HOU_NE,2025,DIV,20,2026-01-18,Sunday,15:00,HOU,16,NE,...,2.0,00-0039163,00-0039851,C.J. Stroud,Drake Maye,DeMeco Ryans,Mike Vrabel,Shawn Smith,BOS00,Gillette Stadium
3025,2025_21_NE_DEN,2025,CON,21,2026-01-25,Sunday,15:00,NE,10,DEN,...,10.0,00-0039851,00-0035264,Drake Maye,Jarrett Stidham,Mike Vrabel,Sean Payton,Alex Kemp,DEN00,Empower Field at Mile High
3026,2025_21_LA_SEA,2025,CON,21,2026-01-25,Sunday,18:30,LA,27,SEA,...,2.0,00-0026498,00-0034869,Matthew Stafford,Sam Darnold,Sean McVay,Mike Macdonald,Clay Martin,SEA00,Lumen Field


In [56]:
regular_games.shape

(2895, 46)

In [57]:
backbone_columns = [
    "game_id",
    "season",
    "week",
    "gameday",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "home_rest",
    "away_rest",
    "location"
]

regular_games[backbone_columns].head(10)

,game_id,season,week,gameday,home_team,away_team,home_score,away_score,home_rest,away_rest,location
0,2015_01_PIT_NE,2015,1,2015-09-10,NE,PIT,28,21,7,7,Home
1,2015_01_IND_BUF,2015,1,2015-09-13,BUF,IND,27,14,7,7,Home
2,2015_01_GB_CHI,2015,1,2015-09-13,CHI,GB,23,31,7,7,Home
3,2015_01_KC_HOU,2015,1,2015-09-13,HOU,KC,20,27,7,7,Home
4,2015_01_CAR_JAX,2015,1,2015-09-13,JAX,CAR,9,20,7,7,Home
5,2015_01_CLE_NYJ,2015,1,2015-09-13,NYJ,CLE,31,10,7,7,Home
6,2015_01_SEA_STL,2015,1,2015-09-13,STL,SEA,34,31,7,7,Home
7,2015_01_MIA_WAS,2015,1,2015-09-13,WAS,MIA,10,17,7,7,Home
8,2015_01_NO_ARI,2015,1,2015-09-13,ARI,NO,31,19,7,7,Home
9,2015_01_DET_SD,2015,1,2015-09-13,SD,DET,33,28,7,7,Home


In [58]:
regular_games[["home_score", "away_score"]].isna().sum()

home_score    0
away_score    0
dtype: int64

In [59]:
(regular_games["home_score"] == regular_games["away_score"]).sum()

np.int64(10)

In [60]:
game_backbone = regular_games.loc[
    regular_games["home_score"] != regular_games["away_score"],
    backbone_columns
    ].copy()

In [61]:
game_backbone.head()

,game_id,season,week,gameday,home_team,away_team,home_score,away_score,home_rest,away_rest,location
0,2015_01_PIT_NE,2015,1,2015-09-10,NE,PIT,28,21,7,7,Home
1,2015_01_IND_BUF,2015,1,2015-09-13,BUF,IND,27,14,7,7,Home
2,2015_01_GB_CHI,2015,1,2015-09-13,CHI,GB,23,31,7,7,Home
3,2015_01_KC_HOU,2015,1,2015-09-13,HOU,KC,20,27,7,7,Home
4,2015_01_CAR_JAX,2015,1,2015-09-13,JAX,CAR,9,20,7,7,Home


In [62]:
game_backbone["home_win"] = (
    game_backbone["home_score"] > game_backbone["away_score"]
).astype(int)

In [63]:
game_backbone[["season", "week", "away_team", "home_team", "away_score", "home_score", "home_win"]].sample(10)

,season,week,away_team,home_team,away_score,home_score,home_win
1052,2018,17,CHI,MIN,24,10,0
1360,2020,2,CAR,TB,17,31,1
1568,2020,16,IND,PIT,24,28,1
1254,2019,13,CLE,PIT,13,20,1
727,2017,14,IND,BUF,7,13,1
2051,2022,11,KC,LAC,30,27,0
1432,2020,7,PIT,TEN,27,24,0
1793,2021,13,WAS,LV,17,15,0
503,2016,16,CIN,HOU,10,12,1
174,2015,12,NE,DEN,24,30,1


In [64]:
game_backbone["home_win"].value_counts()

home_win
1    1575
0    1310
Name: count, dtype: int64

In [65]:
game_backbone["home_win"].value_counts(normalize=True)

home_win
1    0.545927
0    0.454073
Name: proportion, dtype: float64

In [66]:
games.to_csv(
    "../data/raw/schedules_2015_2025.csv",
    index=False
)

game_backbone.to_csv(
    "../data/processed/game_backbone.csv",
    index=False
)

## Creating Pregame Team Features

Machine-learning features must represent information that was available before each game.

To calculate team form, the game-level schedule data will first be transformed into a team-level dataset where each game appears twice: once from each team's perspective.

Initial features will include:

- Recent win percentage
- Recent points scored
- Recent points allowed
- Recent point differential

All rolling statistics will use only games played before the game being predicted.

In [67]:
feature_games = regular_games.dropna(
    subset=["home_score", "away_score"]
).copy()

feature_games["gameday"] = pd.to_datetime(
    feature_games["gameday"]
)

In [68]:
feature_games["home_result_value"] = np.select(
    [
        feature_games["home_score"] > feature_games["away_score"],
        feature_games["home_score"] == feature_games["away_score"]
    ],
    [
        1.0,
        0.5
    ],
    default=0.0
)

feature_games["away_result_value"] = (
    1-feature_games["home_result_value"]
)

In [69]:
home_rows = feature_games[
    [
        "game_id",
        "season",
        "week",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_result_value"
    ]
].rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against",
        "home_result_value": "result_value"
    }
)

home_rows["is_home"] = 1

In [70]:
away_rows = feature_games[
    [
        "game_id",
        "season",
        "week",
        "gameday",
        "away_team",
        "home_team",
        "away_score",
        "home_score",
        "away_result_value"
    ]
].rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against",
        "away_result_value": "result_value"
    }
)

away_rows["is_home"] = 0

In [71]:
team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

In [72]:
team_games["point_diff"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

In [73]:
team_games = team_games.sort_values(
    ["season", "team", "gameday"]
).reset_index(drop=True)

In [74]:
team_games.shape

(5790, 11)

In [75]:
team_games.head(10)

,game_id,season,week,gameday,team,opponent,points_for,points_against,result_value,is_home,point_diff
0,2015_01_NO_ARI,2015,1,2015-09-13,ARI,NO,31,19,1.0,1,12
1,2015_02_ARI_CHI,2015,2,2015-09-20,ARI,CHI,48,23,1.0,0,25
2,2015_03_SF_ARI,2015,3,2015-09-27,ARI,SF,47,7,1.0,1,40
3,2015_04_STL_ARI,2015,4,2015-10-04,ARI,STL,22,24,0.0,1,-2
4,2015_05_ARI_DET,2015,5,2015-10-11,ARI,DET,42,17,1.0,0,25
5,2015_06_ARI_PIT,2015,6,2015-10-18,ARI,PIT,13,25,0.0,0,-12
6,2015_07_BAL_ARI,2015,7,2015-10-26,ARI,BAL,26,18,1.0,1,8
7,2015_08_ARI_CLE,2015,8,2015-11-01,ARI,CLE,34,20,1.0,0,14
8,2015_10_ARI_SEA,2015,10,2015-11-15,ARI,SEA,39,32,1.0,0,7
9,2015_11_CIN_ARI,2015,11,2015-11-22,ARI,CIN,34,31,1.0,1,3


In [76]:
team_group = team_games.groupby(
    ["season", "team"]
)

In [77]:
team_games["win_pct_last5"] = (
    team_group["result_value"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

In [78]:
team_games["points_for_last5"] = (
    team_group["points_for"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

team_games["points_against_last5"] = (
    team_group["points_against"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

team_games["point_diff_last5"] = (
    team_group["point_diff"]
    .transform(
        lambda x: x.shift(1)
                    .rolling(5, min_periods=1)
                    .mean()
    )
)

In [79]:
team_games.loc[
    (team_games["team"] == "CIN") &
    (team_games["season"] == 2025),
    [
        "week",
        "opponent",
        "points_for",
        "points_against",
        "result_value",
        "win_pct_last5",
        "points_for_last5",
        "points_against_last5",
        "point_diff_last5"
    ]
].head(8)

,week,opponent,points_for,points_against,result_value,win_pct_last5,points_for_last5,points_against_last5,point_diff_last5
5348,1,CLE,17,16,1.0,NaN,NaN,NaN,NaN
5349,2,JAX,31,27,1.0,1.000000,17.000000,16.000000,1.0
5350,3,MIN,10,48,0.0,1.000000,24.000000,21.500000,2.5
5351,4,DEN,3,28,0.0,0.666667,19.333333,30.333333,-11.0
5352,5,DET,24,37,0.0,0.500000,15.250000,29.750000,-14.5
5353,6,GB,18,27,0.0,0.400000,17.000000,31.200000,-14.2
5354,7,PIT,33,31,1.0,0.200000,17.200000,33.400000,-16.2
5355,8,NYJ,38,39,0.0,0.200000,17.600000,34.200000,-16.6


## Loading Weekly Team Performance Data

The schedule dataset provides the game backbone, but it does not contain enough football-performance information for a meaningful prediction model.

Weekly team statistics will now be loaded so that offensive performance can be measured before each game.

These statistics will later be converted into rolling pregame features, just like the recent-form features above.

In [80]:
weekly_stats = nfl.load_team_stats(
    seasons=seasons,
    summary_level="week"
).to_pandas()

In [81]:
weekly_stats.shape

(6056, 138)

In [85]:
weekly_stats.columns.tolist()

['season',
 'week',
 'team',
 'season_type',
 'game_id',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'receiving_10',
 'receiving_16',
 'receiving_20',
 'receiving_40',
 'special_teams_tds',
 'def_ta

In [86]:
weekly_stats.head()

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,...,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
0,2015,1,ARI,REG,2015_01_NO_ARI,NO,19,32,307,3,...,158,2,1,1,0,1,1,2,0,156
1,2015,1,ATL,REG,2015_01_PHI_ATL,PHI,23,34,298,2,...,231,3,1,0,0,0,3,25,0,206
2,2015,1,BAL,REG,2015_01_BAL_DEN,DEN,18,32,117,0,...,310,2,2,0,0,1,3,20,0,290
3,2015,1,BUF,REG,2015_01_IND_BUF,IND,14,19,195,1,...,252,2,0,0,1,2,2,1,0,231
4,2015,1,CAR,REG,2015_01_CAR_JAX,JAX,18,31,175,1,...,179,2,0,0,1,2,1,9,0,150


In [87]:
weekly_stats["season_type"].value_counts()

season_type
REG     5790
POST     266
Name: count, dtype: int64

In [89]:
weekly_reg = weekly_stats.loc[
    weekly_stats["season_type"] == "REG"
].copy()

In [90]:
weekly_reg.shape

(5790, 138)

## Creating Team Efficiency Metrics

Raw totals such as passing yards and rushing yards can be affected by game volume and game script.

To better measure team quality, several efficiency statistics will be derived from the weekly data.

Initial efficiency metrics include:

- Passing EPA per dropback
- Rushing EPA per carry
- Sack rate
- Turnovers

In [91]:
weekly_reg["dropbacks"] = (
    weekly_reg["attempts"]
    + weekly_reg["sacks_suffered"]
)

In [92]:
weekly_reg["pass_epa_per_dropback"] = (
    weekly_reg["passing_epa"]
    / weekly_reg["dropbacks"]
)

weekly_reg["rush_epa_per_carry"] = (
    weekly_reg["rushing_epa"]
    / weekly_reg["carries"]
)

weekly_reg["sack_rate"] = (
    weekly_reg["sacks_suffered"]
    / weekly_reg["dropbacks"]
)

weekly_reg["turnovers"] = (
    weekly_reg["passing_interceptions"] 
    + weekly_reg["fumbles_lost_total"]
)

In [93]:
initial_stats = [
    "season",
    "week",
    "team",
    "opponent_team",
    "game_id",
    "attempts",
    "dropbacks",
    "passing_epa",
    "pass_epa_per_dropback",
    "passing_cpoe",
    "carries",
    "rushing_epa",
    "rush_epa_per_carry",
    "sack_rate",
    "turnovers"
]

In [94]:
weekly_reg[initial_stats].head(10)

,season,week,team,opponent_team,game_id,attempts,dropbacks,passing_epa,pass_epa_per_dropback,passing_cpoe,carries,rushing_epa,rush_epa_per_carry,sack_rate,turnovers
0,2015,1,ARI,NO,2015_01_NO_ARI,32,32,16.250638,0.507832,-2.395354,25,-8.709702,-0.348388,0.000000,1
1,2015,1,ATL,PHI,2015_01_PHI_ATL,34,35,1.314335,0.037552,2.046971,35,-8.790971,-0.251171,0.028571,2
2,2015,1,BAL,DEN,2015_01_BAL_DEN,32,34,-17.578055,-0.517002,-8.127491,23,-4.362853,-0.189689,0.058824,2
3,2015,1,BUF,IND,2015_01_IND_BUF,19,19,9.972638,0.524876,5.194864,36,-2.539012,-0.070528,0.000000,0
4,2015,1,CAR,JAX,2015_01_CAR_JAX,31,33,-0.279408,-0.008467,-6.241385,35,-5.129245,-0.146550,0.060606,1
5,2015,1,CHI,GB,2015_01_GB_CHI,36,38,0.229273,0.006034,-8.881223,33,7.547915,0.228725,0.052632,1
6,2015,1,CIN,LV,2015_01_CIN_OAK,34,34,13.905390,0.408982,11.322684,31,0.289956,0.009353,0.000000,0
7,2015,1,CLE,NYJ,2015_01_CLE_NYJ,32,35,-4.041235,-0.115464,-7.247922,28,-8.630165,-0.308220,0.085714,5
8,2015,1,DAL,NYG,2015_01_NYG_DAL,45,45,9.175033,0.203890,8.768243,23,-2.428181,-0.105573,0.000000,3
9,2015,1,DEN,BAL,2015_01_BAL_DEN,40,44,-12.047624,-0.273810,-7.397209,25,-4.829149,-0.193166,0.090909,1


In [96]:
weekly_reg[
    [
        "pass_epa_per_dropback",
        "passing_cpoe",
        "rush_epa_per_carry",
        "sack_rate",
        "turnovers"
    ]
].describe()

,pass_epa_per_dropback,passing_cpoe,rush_epa_per_carry,sack_rate,turnovers
count,5790.000000,5790.000000,5790.000000,5790.000000,5790.000000
mean,0.045113,0.488059,-0.064566,0.065894,1.322971
std,0.314622,8.656397,0.210623,0.048655,1.198119
min,-2.370988,-38.760544,-1.734321,0.000000,0.000000
25%,-0.160487,-5.089961,-0.190937,0.030303,0.000000
50%,0.046226,0.548805,-0.057668,0.058824,1.000000
75%,0.249593,6.270847,0.073494,0.093023,2.000000
max,1.444429,28.640443,0.892451,0.423077,8.000000


## Deriving Defensive Performance

The weekly team statistics primarily describe offensive production.

A team's defensive performance can be derived from the offensive performance of its opponent in the same game.

For example, if an opponent produces +0.20 passing EPA per dropback, the defense allowed +0.20 passing EPA per dropback.

The weekly dataset will therefore be joined to a transformed copy of itself using the game ID and opponent.

In [97]:
opponent_stats = weekly_reg[
    [
        "game_id",
        "team",
        "opponent_team",
        "pass_epa_per_dropback",
        "passing_cpoe",
        "rush_epa_per_carry",
        "sack_rate",
        "turnovers"
    ]
].copy()

opponent_stats = opponent_stats.rename(
    columns={
        "team": "opponent_team",
        "opponent_team": "team",
        "pass_epa_per_dropback": "pass_epa_allowed_per_dropback",
        "passing_cpoe": "passing_cpoe_allowed",
        "rush_epa_per_carry": "rush_epa_allowed_per_carry",
        "sack_rate": "def_sack_rate",
        "turnovers": "takeaways"
    }
)

team_performance = weekly_reg.merge(
    opponent_stats,
    on=["game_id", "team", "opponent_team"],
    how="left",
    validate="one_to_one"
)

In [98]:
team_performance[
    [
        "season",
        "week",
        "team",
        "opponent_team",
        "pass_epa_per_dropback",
        "pass_epa_allowed_per_dropback",
        "rush_epa_per_carry",
        "rush_epa_allowed_per_carry",
        "sack_rate",
        "def_sack_rate",
        "turnovers",
        "takeaways"
    ]
].head(10)

,season,week,team,opponent_team,pass_epa_per_dropback,pass_epa_allowed_per_dropback,rush_epa_per_carry,rush_epa_allowed_per_carry,sack_rate,def_sack_rate,turnovers,takeaways
0,2015,1,ARI,NO,0.507832,0.036596,-0.348388,-0.187498,0.000000,0.040000,1,1
1,2015,1,ATL,PHI,0.037552,0.088731,-0.251171,0.049152,0.028571,0.000000,2,2
2,2015,1,BAL,DEN,-0.517002,-0.273810,-0.189689,-0.193166,0.058824,0.090909,2,1
3,2015,1,BUF,IND,0.524876,0.044660,-0.070528,0.103916,0.000000,0.039216,0,3
4,2015,1,CAR,JAX,-0.008467,-0.344942,-0.146550,-0.042704,0.060606,0.111111,1,3
5,2015,1,CHI,GB,0.006034,0.582160,0.228725,0.160186,0.052632,0.000000,1,0
6,2015,1,CIN,LV,0.408982,-0.082221,0.009353,-0.457011,0.000000,0.044444,0,2
7,2015,1,CLE,NYJ,-0.115464,0.430251,-0.308220,-0.156182,0.085714,0.000000,5,1
8,2015,1,DAL,NYG,0.203890,-0.000685,-0.105573,0.120182,0.000000,0.027027,3,0
9,2015,1,DEN,BAL,-0.273810,-0.517002,-0.193166,-0.189689,0.090909,0.058824,1,2


In [102]:
team_performance = team_performance.sort_values(
    ["season", "team", "week"]
).reset_index(drop=True)

performance_features = [
    "pass_epa_per_dropback",
    "passing_cpoe",
    "rush_epa_per_carry",
    "sack_rate",
    "turnovers",
    "pass_epa_allowed_per_dropback",
    "passing_cpoe_allowed",
    "rush_epa_allowed_per_carry",
    "def_sack_rate",
    "takeaways"
]

for feature in performance_features:
    team_performance[f"{feature}_last5"] = (
        team_performance
        .groupby(["season", "team"])[feature]
        .transform(
            lambda x: x.shift(1)
                        .rolling(5, min_periods=1)
                        .mean()
        )
    )

In [103]:
team_performance.loc[
    (team_performance["team"] == "CIN") &
    (team_performance["season"] == 2025),
    [
        "week",
        "opponent_team",

        "pass_epa_per_dropback_last5",
        "pass_epa_allowed_per_dropback_last5",

        "sack_rate_last5",
        "def_sack_rate_last5",

        "turnovers_last5",
        "takeaways_last5"
    ]
].head(8)

,week,opponent_team,pass_epa_per_dropback_last5,pass_epa_allowed_per_dropback_last5,sack_rate_last5,def_sack_rate_last5,turnovers_last5,takeaways_last5
5348,1,CLE,NaN,NaN,NaN,NaN,NaN,NaN
5349,2,JAX,-0.012754,-0.039194,0.115385,0.042553,0.000000,2.000000
5350,3,MIN,-0.025421,0.038606,0.088942,0.032905,1.500000,2.000000
5351,4,DEN,-0.224577,0.095127,0.102306,0.058973,2.666667,1.333333
5352,5,DET,-0.260885,0.150392,0.103515,0.044230,2.000000,1.250000
5353,6,GB,-0.250451,0.213263,0.092336,0.063955,2.200000,1.200000
5354,7,PIT,-0.219655,0.275591,0.073607,0.062852,2.200000,1.000000
5355,8,NYJ,-0.150225,0.330615,0.069270,0.058201,1.600000,1.000000
